In [1]:
import pandas as pd

def prism_algorithm(df, target_col, target_value, max_rules=5):
    data = df.copy()
    all_rules = []
    
    # Enquanto houver casos da classe alvo
    while len(data[data[target_col] == target_value]) > 0 and len(all_rules) < max_rules:
        current_rule = []
        subset = data.copy()
        
        while True:
            best_p = -1
            best_cond = None
            
            # Testa cada combinação Atributo-Valor
            for col in [c for c in subset.columns if c != target_col]:
                for val in subset[col].unique():
                    temp_subset = subset[subset[col] == val]
                    if len(temp_subset) == 0: continue
                    
                    # Precisão: P(Classe | Condição)
                    p = len(temp_subset[temp_subset[target_col] == target_value]) / len(temp_subset)
                    
                    if p > best_p:
                        best_p = p
                        best_cond = (col, val)
            
            if best_cond:
                current_rule.append(best_cond)
                subset = subset[subset[best_cond[0]] == best_cond[1]]
                if best_p == 1.0: break # Regra 100% pura encontrada
            else: break
            
        all_rules.append(current_rule)
        # Remove instâncias cobertas para encontrar a próxima regra
        mask = pd.Series([True] * len(data), index=data.index)
        for c, v in current_rule:
            mask &= (data[c] == v)
        data = data[~mask]
        
    return all_rules

In [ ]:
# Carregar dados
df_hr = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Discretização: Transformando números em categorias
df_hr['Renda_Nivel'] = pd.qcut(df_hr['MonthlyIncome'], q=3, labels=['Baixa', 'Media', 'Alta'])
df_hr['Satisfacao'] = df_hr['JobSatisfaction'].map({1:'Baixa', 2:'Media', 3:'Alta', 4:'Muito_Alta'})

# Seleção de atributos e execução
cols = ['OverTime', 'MaritalStatus', 'Renda_Nivel', 'Satisfacao', 'Attrition']
regras_hr = prism_algorithm(df_hr[cols], 'Attrition', 'Yes')

print("=== BASE DE CONHECIMENTO PRISM: GESTÃO DE RH ===")
for i, r in enumerate(regras_hr):
    cond = " E ".join([f"({c} = {v})" for c, v in r])
    print(f"Regra {i+1}: SE {cond} ENTÃO Attrition = 'Yes'")